# 10_explainability_and_trade_journal.ipynb

This notebook generates SHAP feature importances and produces human-readable explanation cards for trading decisions.

### Objectives:
1. Load the trade journal and test dataset.
2. Calculate SHAP values for the trained LightGBM model.
3. Analyze global and local feature importance.
4. Generate human-readable decision cards for selected trades.
5. Save explainability reports under `reports/explainability/`.

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Git Integration - Pull latest code from GitHub
import os
PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
GITHUB_REPO_URL = "https://github.com/umutergul74/TradeBot.git"

print("Pulling latest code from GitHub...")
os.chdir(PROJECT_ROOT)
# Initialize git and pull
!git init -b main
!git remote remove origin 2>/dev/null || true
!git remote add origin {GITHUB_REPO_URL}
!git fetch origin
!git reset --hard origin/main
print("Codebase updated to latest GitHub commit!")

In [ ]:
# Auto-install dependencies if any key package is missing
try:
    import ccxt
    import vectorbt
    import optuna
    import catboost
    import ta
    import shap
except ImportError:
    print("Some dependencies are missing. Installing from requirements.txt...")
    requirements_path = os.path.join(PROJECT_ROOT, "requirements.txt")
    !pip install -q -r {requirements_path}
    print("Installation complete!")

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Load Dataset & Models

In [ ]:
from utils.config import load_global_config
from utils.io_utils import load_parquet
from models.model_registry import ModelRegistry

config = load_global_config(PROJECT_ROOT)
symbol = config.get("data", "symbol", "BTCUSDT")

# Load out-of-sample data
features_path = os.path.join(PROJECT_ROOT, "data", "feature_store", f"{symbol}_5m_features.parquet")
df_features = load_parquet(features_path)
labels_path = os.path.join(PROJECT_ROOT, "data", "labels", f"{symbol}_5m_labels.parquet")
df_labels = load_parquet(labels_path)
df_model = df_features.loc[df_labels.index].copy()

n_samples = len(df_model)
test_idx = int(n_samples * 0.8)
df_test = df_model.iloc[test_idx:].copy()

# Load model
registry = ModelRegistry(PROJECT_ROOT)
model_wrapper = registry.load_model(f"{symbol}_lgbm_model")
calibrator = registry.load_model(f"{symbol}_lgbm_calibrator")

## 3. Generate SHAP Explanations

In [ ]:
import shap

exclude_cols = ["label", "meta_label", "timestamp", "open", "high", "low", "close", "volume", "taker_buy_base_volume"]
features = [col for col in df_model.columns if col not in exclude_cols]

X_test = df_test[features].values

# Initialize SHAP explainer
explainer = shap.TreeExplainer(model_wrapper.model)
shap_values = explainer.shap_values(X_test)

# Note: For binary classification, shap_values is a list of two arrays (class 0, class 1) or a single array.
if isinstance(shap_values, list):
    shap_val_class1 = shap_values[1]
else:
    shap_val_class1 = shap_values
    
print("SHAP calculation completed.")

## 4. Generate Human-Readable Decision Cards

In [ ]:
from explainability.signal_explainer import generate_trade_explanation

# Create a mock trade to demonstrate the explanation generator
sample_trade = {
    "decision": "LONG",
    "confidence": 0.78,
    "setup": "Sell-side liquidity sweep + bullish FVG mitigation + bullish CVD divergence",
    "htf_bias": "Bullish",
    "entry": 64120.00,
    "stop": 63780.00,
    "tp1": 64800.00,
    "tp2": 65500.00,
    "rr": 2.1,
    "ev": "Positive",
    "reasons": [
        "Price swept the previous Asia session low and reclaimed the range.",
        "A bullish displacement candle created a 5m FVG.",
        "Price returned into the FVG midpoint while still in discount.",
        "CVD formed a bullish divergence while price made a lower low.",
        "Higher timeframe structure remains bullish.",
        "The trade passes the minimum expected value and risk/reward filters."
    ]
 }

card_text = generate_trade_explanation(sample_trade)
print(card_text)

# Save the card text to Drive
card_path = os.path.join(PROJECT_ROOT, "reports", "explainability", "sample_trade_decision.txt")
os.makedirs(os.path.dirname(card_path), exist_ok=True)
with open(card_path, "w") as f:
    f.write(card_text)

## Summary of Explainability Reports

We completed:
1. SHAP feature contribution analysis on the out-of-sample test dataset.
2. Generated a human-readable explanation card saved to `reports/explainability/sample_trade_decision.txt`.

**Next Step**: Run [11_dashboard_and_visual_analysis.ipynb](file:///content/drive/MyDrive/btcusdt_quant_research/notebooks/11_dashboard_and_visual_analysis.ipynb) to view interactive charts and performance dashboards.

In [ ]:
# Auto-disconnect Colab runtime to save compute units
AUTO_DISCONNECT = False  # Set to True to enable automatic shutdown
if AUTO_DISCONNECT:
    print("Disconnecting Colab runtime...")
    from google.colab import runtime
    runtime.unassign()